# Bias-Variance and Cross-Validation Practice

**Topic:** Statistical Learning &nbsp;·&nbsp; **Difficulty:** Very Hard

## The data (created in the next cell)

- `x_train` `(100,)`, `y_train`: noisy samples of a smooth nonlinear function on [-1, 1]. That's all the labeled data you get.
- `x_test` `(400,)`: predict `y` here. `y_test` is used by the evaluation cell.

## Your task

1. For each degree in 1…12 and each ridge alpha in a log grid, estimate the held-out R² with k-fold cross-validation **on the training set only** (`cross_val_score` or a manual fold loop).
2. Select the (degree, alpha) pair with the best CV score, and optionally plot the CV curve against the degree.
3. Refit on all 100 points and set `predictions` at `x_test`.

## Evaluation

Test R² must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def truth(x):
    return 2.5 * x ** 3 + 0.9 * np.sin(4 * x) - 0.5 * x

def make(n, seed):
    r = np.random.default_rng(seed)
    x = r.uniform(-1, 1, n)
    return x, truth(x) + r.normal(0, 0.18, n)

x_train, y_train = make(100, 161)
x_test, y_test = make(400, 162)
print(f"train {x_train.shape}: small on purpose, cross-validate!")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your model and try again.")


## Solution

Grid-search degree × alpha with cross-validation on the training set only: build Vandermonde features per degree, score ridge models with 5-fold CV, keep the best pair, and refit on all training points.

```python
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

def feats(x, degree):
    return np.vander(np.asarray(x), degree + 1, increasing=True)

best = None
for degree in range(1, 13):
    for alpha in np.logspace(-6, 1, 8):
        score = cross_val_score(Ridge(alpha=alpha), feats(x_train, degree), y_train, cv=5).mean()
        if best is None or score > best[0]:
            best = (score, degree, alpha)

score, degree, alpha = best
print(f"chosen degree {degree}, alpha {alpha:.2g} (CV R^2 {score:.3f})")
model = Ridge(alpha=alpha).fit(feats(x_train, degree), y_train)
predictions = model.predict(feats(x_test, degree))
```
